# HETDEX Source Catalog 2: Catalog description and access

In [1]:
# Required python moduldes for opening FITS files and plotting
import numpy as np
import os.path as op
import os
import glob

import matplotlib.pyplot as plt
from matplotlib import gridspec

from astropy.io import fits
from astropy.table import Table, hstack
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS

from astropy.visualization import make_lupton_rgb, ZScaleInterval

In [2]:
%matplotlib inline

In [3]:
#Download the catalog to your local directory if you are not on the HETDEX JupyterHub

In [4]:
# fill in the path to the HETDEX Source Catalog 2 files 
path_to_cat = '/home/jovyan/Hobby-Eberly-Public/HETDEX/pdr/pdr1/hetdex_source_catalog_2/'
version = 'v1.3' # Change to latest version 

# switch to True if you are not on the HETDEX JupyterHub to download catalog
remote = False

if remote:
    # Put catalog in desired working directory in directory hetdex_source_catalog_2
    path_to_cat = "/home/jovyan/work/pdr1/hetdex_source_catalog_2"

    os.makedirs(path_to_cat, exist_ok=True)
    url = "http://web.corral.tacc.utexas.edu/hetdex/HETDEX/pdr/pdr1/hetdex_source_catalog_2/"

    !wget -r -np -nd -A "*.fits" -P "$path_to_cat" "$url"

## Catalog Organization

Two catalogs make up HETDEX Source Catalog 2:
    
    1. The Source Observation Table: hetdex_sc2_vX.dat/.fits
       With SPECTRA arrays included: hetdex_sc2_spec_vX.fits
       
        One row per source observation. The table provides basic coordinates/redshift/source information for each observation of a unique astornomical source. The larger file hetdex_sc2_spec_vX.fits contains the same info from the first table plus addition data units of spectral array data.
        
    2. The Detection Information Table: hetdex_sc2_detinfo_vX.fits
        One row per line or continuum detection. Bright sources can be comprised of multiple line or continuum emission. This catalog provides specific detection information such as line parameter info (S/N, line flux, line width), observational data and instrument details.

## How to Open the Source Observation Table without Spectra

Multiple formats are provided for the source table.

In [5]:
source_table = Table.read(op.join( path_to_cat, 'hetdex_sc2_{}.fits'.format(version) ) )                       

In [8]:
print(f"{len(source_table):,} rows\n")

for c in source_table.colnames:
    col = source_table[c]
    desc = col.description if col.description else ""
    print(f"{c:25} {str(col.dtype):8} {str(col.unit):20} {desc}")

1,084,831 rows

source_name               |S26     None                 HETDEX IAU designation
source_id                 >i8      None                 HETDEX Source Identifier
shotid                    >i8      None                 integer represent observation ID: int( date+obsid)
RA                        >f4      deg                  source_id right ascension (ICRS deg)
DEC                       >f4      deg                  source_id declination (ICRS deg)
gmag                      >f4      None                 sdss-g magnitude measured in HETDEX spectrum
Av                        >f4      None                 applied dust correction in V band
z_hetdex                  >f4      None                 HETDEX spectroscopic redshift
z_hetdex_src              |S8      None                 HETDEX spectroscopic redshift source
z_hetdex_conf             >f4      None                 0 to 1 confidence HETDEX spectroscopic redshift source
source_type               |S10     None               

## How to Open the Source Observation Table with Spectra

If spectral data is desired, a fits file contains spectral array data matching each row of the Source Observation Table

In [9]:
hdu = fits.open( op.join( path_to_cat, 'hetdex_sc2_spec_{}.fits'.format(version)))

In [10]:
# The source table can also be accessed by astropy Table class or through HDU1
#source_table = Table.read( op.join( path_to_cat, 'hetdex_sc2_spec_{}.fits'.format(version)))

In [11]:
hdu.info()

Filename: /home/jovyan/Hobby-Eberly-Public/HETDEX/pdr/pdr1/hetdex_source_catalog_2/hetdex_sc2_spec_v1.3.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  INFO          1 BinTableHDU     93   1084831R x 34C   [26A, K, K, E, E, E, E, E, 8A, E, 10A, K, K, 12A, E, E, E, E, K, E, E, E, D, D, D, D, E, E, E, E, E, E, E, E]   
  2  SPEC          1 ImageHDU        10   (1084831, 1036)   float32   
  3  SPEC_ERR      1 ImageHDU         9   (1084831, 1036)   float32   
  4  WAVELENGTH    1 ImageHDU         8   (1036,)   float32   


In [12]:
spec = hdu['SPEC'].data
spec_err = hdu['SPEC_ERR'].data
wave_rect = hdu['WAVELENGTH'].data

In [13]:
#spec unit is:
u.Unit( hdu['SPEC'].header['BUNIT'])

Unit("1e-17 erg / (Angstrom s cm2)")

In [14]:
# wavelength unit is:
u.Unit( hdu['WAVELENGTH'].header['BUNIT'])

Unit("Angstrom")

### Query by source type example

In [15]:
sel_lae = source_table['source_type'] == 'lae'
print('There are {} LAES in the catalog'.format(np.sum(sel_lae)))

There are 406320 LAES in the catalog


In [16]:
source_ids_lae = source_table['source_id'][sel_lae]

In [17]:
# let's plot up all LAEs observed after June 2024. We can use the shotid colun
# as it is organized based on date

In [18]:
sel_date = source_table['shotid'] > 20240600000
sel_flux = source_table['flux_lya'] > 15

In [19]:
sel =  sel_lae & sel_date & sel_flux

In [20]:
np.shape( spec[:, sel])

(1036, 3337)

In [21]:
lae_spec_table = hstack([
    source_table[sel],
    spec[:, sel].T,
    spec_err[:, sel].T
])

In [22]:
lae_spec_table.rename_column('col0_2', 'spec')
lae_spec_table.rename_column('col0_3', 'spec_err')

In [23]:
lae_spec_table.sort('z_hetdex')

In [24]:
plt.figure(figsize=(4, 10))
plt.imshow( lae_spec_table['spec'], aspect=0.1, vmin=-0.01, vmax=2)
plt.title('HETDEX LAE spectra in increasing redshift order')
plt.xlabel('wavelength/z_hetdex')
plt.ylabel('HETDEX 1D LAE spectra')

Text(0, 0.5, 'HETDEX 1D LAE spectra')

### Query by Sky Coordinate Example

In [25]:
# Create array of coordinates for all HETDEX source members
source_coords = SkyCoord(ra = source_table['RA'], dec= source_table['DEC'])

In [26]:
coord = SkyCoord(ra=220.21432*u.deg, dec=52.095898*u.deg)

In [27]:
sel_match = np.where(  source_coords.separation(coord) < 1.*u.arcsec)[0]

### Access and Plot Spectra

In [28]:
redshift = source_table['z_hetdex'][sel_match][0]

lya_wave = 1216* (redshift + 1)
source_id = source_table['source_id'][sel_match][0]
name = source_table['source_name'][sel_match][0]
plt.plot( wave_rect, spec[:, sel_match[0]])

plt.xlim(lya_wave-50, lya_wave+50)
plt.ylabel(r'spec (10$^-17$ ergs/s/cm$^2$/$\AA$)')
plt.xlabel(r'wavelength ($\AA$)')
plt.title('{}  source_id={}'.format(name, source_id))

Text(0.5, 1.0, 'HETDEX J144051.41+520545.3  source_id=5020000432307')

## How to Open the Detection Info Table

In [29]:
det_table = Table.read(op.join(path_to_cat, 'hetdex_sc2_detinfo_{}.fits'.format(version)))

In [31]:
# Column Info

print(f"{len(det_table):,} rows\n")

print(f"{'Column':25} {'dtype':8} {'unit':20} Description")
print("-"*90)

for c in det_table.colnames:
    col = det_table[c]
    desc = getattr(col, "description", "")
    print(f"{c:25} {str(col.dtype):8} {str(col.unit):20} {desc}")

3,296,055 rows

Column                    dtype    unit                 Description
------------------------------------------------------------------------------------------
source_id                 >i8      None                 HETDEX Source Identifier
source_name               |S26     None                 HETDEX IAU designation
RA                        >f4      deg                  source_id right ascension (ICRS deg)
DEC                       >f4      deg                  source_id declination (ICRS deg)
z_hetdex                  >f4      None                 HETDEX spectroscopic redshift
z_hetdex_src              |S8      None                 HETDEX spectroscopic redshift source
z_hetdex_conf             >f4      None                 0 to 1 confidence HETDEX spectroscopic redshift source
source_type               |S10     None                 options are 'star', 'lae', 'agn', 'lzg', 'oii', 'none'
detectid                  >i8      None                 emission line or detection

## Plot up all detections for a single source_id

Let's query the detection info table for a relatively nearby, bright galaxy that is composed of serveral line detections to demonstrate the content of the Detection Info Table. Nearby galaxies and bright sources such as AGN can be composed of many detections. This is because line emission at different spatial regions and wavelengths will result in multiple detections in the detection search. The source will also likely have a complementary continuum detetion if it is bright (g < 21)

In [32]:
sel_big_oii = (det_table['source_type'] == 'oii') & (det_table['major'] > 6) 
sel_center_ifu = (np.abs( det_table['x_ifu']) < 5) & (np.abs(det_table['y_ifu']) < 5)
selected_det = (det_table['selected_det'] == True) & (det_table['n_members'] >6)

In [33]:
sel = sel_big_oii & sel_center_ifu & selected_det

print(' There are {} matches '.format(np.sum(sel)))

 There are 174 matches 


In [34]:
# Pick random object in list and plot up the source_id

index = np.where(sel)[0][0]

sid = det_table['source_id'][index]
 
coords = SkyCoord(ra=det_table['RA'][index]*u.deg, dec=det_table['DEC'][index]*u.deg)

In [35]:
# Get Imaging data from Legacy Survey API
fits_file = 'https://www.legacysurvey.org/viewer/fits-cutout?ra={}&dec={}&layer=ls-dr9&width=80&height=80&pixscale=0.25&bands=grz'.format(coords.ra.deg, coords.dec.deg)
hdu_ls = fits.open(fits_file)
wcs_ls = WCS( hdu_ls[0].header).dropaxis(2)


In [36]:
redshift = det_table['z_hetdex'][index]
source_type = det_table['source_type'][index]

grp = det_table[ det_table['source_id'] == sid]
grp.sort('gmag')

In [37]:
# Get Spectrum from source_table
hdu = fits.open(op.join(path_to_cat, 'hetdex_sc2_spec_{}.fits'.format(version)))
source_table = hdu['INFO'].data
spec = hdu['SPEC'].data
sel_source = np.where( source_table['source_id'] == sid)[0][0]
spectra = spec[:, sel_source]

In [38]:
plt.figure(figsize=(13,4))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.2,3])

ax1 = plt.subplot(gs[0], projection=wcs_ls)
ax1.set_aspect('auto')
im_zscale = ZScaleInterval(contrast=0.5, krej=1.1)
im_vmin, im_vmax = im_zscale.get_limits(values=hdu_ls[0].data[2])
plt.imshow(hdu_ls[0].data[2], origin='lower', cmap=plt.get_cmap('gray_r'), vmin=im_vmin, vmax=im_vmax  )

sel_line = (grp["det_type"] == "line")
if np.sum(sel_line) >= 1:
    plt.scatter(
        grp["RA_det"][sel_line],
        grp["DEC_det"][sel_line],
        transform=ax1.get_transform("world"),
        marker="x",
        color="orange",
        linewidth=2,
        s=50,
#        zorder=100,
        label="line emission",
    )

sel_cont = grp["det_type"] == "cont"
if np.sum(sel_cont) >= 1:
    plt.scatter(
        grp["RA_det"][sel_cont],
        grp["DEC_det"][sel_cont],
        transform=ax1.get_transform("world"),
        marker="x",
        color="green",
        linewidth=2,
        s=50,
        label="continuum",
    )
lon = ax1.coords[0]
lat = ax1.coords[1]
lon.set_axislabel('RA', minpad=0.5)
#lat.set_axislabel('Dec', minpad=-0.6)
lat.set_axislabel('Dec', minpad=0.3)
lon.set_ticklabel(exclude_overlapping=True)
    
ax2 = plt.subplot(gs[1])

plt.plot(wave_rect, spectra, linewidth=1.2, color='tab:blue')#, yerr=spec_table['spec1d_err'])
plt.xlabel(r'$\lambda$ ($\AA$)')
plt.ylabel(r'f$_\lambda$ (10$^{-17}$ ergs/s/cm$^2$/$\AA$)')
plt.xlim(3540, 5450)

selw = (wave_rect > 3540) & (wave_rect < 5450)
y2 = np.max(spectra[selw])
y1 = np.min(spectra[selw])

if y1 > 0:
    y1 = 0

# plot all emission lines detected in det_table related to the source_id

for line in np.array( np.unique(grp['line_id'])):
    
    if line == b'n/a':
        continue
    
    sel_line = grp['line_id'] == line   

    plt.bar(grp['wave'][sel_line][0], height=2*y2, width=30, bottom=y1, color='orange', alpha=0.3)
    
    label = '{}  [{}]'.format( grp['detectid'][sel_line][0], line)
    if np.isfinite( grp['wave'][sel_line][0]):
        plt.text(grp['wave'][sel_line][0]-70, 0.1*y2, label, rotation=90, fontsize=10)

plt.axhline(0, color='tab:grey', linestyle='dashed')
plt.text(0.05, 0.7, 'z={:6.4f}'.format(redshift), transform=ax2.transAxes, color='tab:red', fontsize=20)
plt.text(0.05, 0.85, 'source_id={}'.format(sid), transform=ax2.transAxes, fontsize=14, color='black')
plt.ylim(y1,1.1*y2)
plt.subplots_adjust(wspace=0.3, hspace=0)
plt.tight_layout()

## ELiXer Reports

An important diagnostic for HETDEX detections is provided by the **ELiXer reports**, which are generated for every curated detection in the HETDEX catalog. A full description of these reports is given in the Appendix of Davis et al. (2023).

ELiXer reports summarize ancillary imaging and spectroscopic information at the location of each detection. They include:
- Imaging cutouts from available surveys
- Cutouts of the amplifier array showing the highest-weight fibers used in the PSF extraction
- The full HETDEX PSF-extracted spectrum with associated error
- A zoomed view of the spectral region around the detected line

These reports are used for visual validation of HETDEX detections.

This release includes **ELiXer reports for all detections** in the full supplemental HETDEX detection catalog (Appendix: Detection Info Table), including every detection in the main HPSC2 catalog.

In [39]:
from IPython.display import Image, display

In [40]:
# plot up all detectids from previous example

detectids = grp['detectid'] 

for det in detectids:
    prefix = str(det)[:5]

    url = f"https://web.corral.tacc.utexas.edu/hetdex/HETDEX/pdr/pdr1/detect/elixer/{prefix}/{det}.jpg"

    display(Image(url=url, width=900))

## References

* Davis, D., Gebhardt, K., Mentuch Cooper, E., et al. 2023, ApJ, 946, 86, doi: 10.3847/1538-4357/acb0ca
* Dey, A., Schlegel, D.J., Lang, D., et al. 2019, ApJ, 157, 168. doi:10.3847/1538-3881/ab089d LEGACY SURVEY: we use the LS API to obtain a sky cutout at the HETDEX source
https://www.legacysurvey.org. 
* Gebhardt, K., Mentuch Cooper, E., Ciardullo, R., et al. 2021, ApJ, 923, 217. doi:10.3847/1538-4357/ac2e03
* Mentuch Cooper, E., Gebhardt, K., Davis, D. et al. 2023, ApJ,  943, 177 doi: 10.3847/1538-4357/aca962 